# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [10]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env
import sys
sys.path.append('../05_src/')


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [12]:
import os
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

In [5]:
API_KEY = os.getenv('API_GATEWAY_KEY')
print(API_KEY)

QWsuGaGHPI50h4eNruqgPUPcqGLp2BQI


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
# Set up Environment Variables
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [1]:
# Load PDF file with LangChain
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "../02_activities/documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))


26


In [2]:
#join collection of pages
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


In [3]:
# Inspect string content and metadata
print(f"{docs[0].page_content[:250]}\n")
print(docs[0].metadata)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 


{'source': '../02_activities/documents/ai_report_2025.pdf', 'page': 0}


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [47]:
system_prompt = "You are an expert in organizational behavior who speaks in formal academic writing."

In [48]:
prompt = f"""
    You are an organizational behavior professor. 
    Given the following context from a document, do the following:
    
    1. Identify the document's title and author(s).
    2. Generate a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. Summarize concisely and in a succinct manner the main points of this document in no longer than 1000 tokens.
        
    The document is the following: 
    <docs>
    {document_text}
    </docs>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Number of Stories: <number_of_stories>
    Summary: <summary>
"""

In [49]:
# Model selection  
response = client.responses.create(
    model = "gpt-4o-mini", 
    instructions = system_prompt,
    input = prompt
)

In [50]:
from IPython.display import display, Markdown
display(Markdown(response.output_text))
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)


**Title:** The GenAI Divide: State of AI in Business 2025  
**Author:** MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)  
**Number of Stories:** 0

**Summary:** This document offers critical insights into the current state of generative AI (GenAI) adoption within enterprises, revealing a significant gap between high adoption rates and low transformative impact—termed the "GenAI Divide." Despite substantial investments in GenAI, only 5% of organizations realize meaningful returns, primarily due to technological and organizational misalignments. The report identifies key patterns underlying the divide, such as the lack of adaptive learning in AI systems, reliance on generic tools, skewed investment towards visible areas like sales over potentially higher-ROI back-office functions, and a prevalent "shadow AI" economy where employees utilize personal AI tools more effectively than formal corporate initiatives. Successful organizations leverage external partnerships, demand specific capabilities from vendors, and decentralize efforts to foster innovation. The document outlines strategies for enterprises to successfully cross the GenAI Divide, emphasizing the importance of customizing solutions and embedding systems to meet organizational needs. Furthermore, it emphasizes that significant ROI often lies in back-office automation rather than front-facing functions, thereby urging organizations to reevaluate their AI deployment strategies and partnerships to thrive within an increasingly competitive landscape.

Input tokens: 10878
Output tokens: 278


In [51]:
# Outpout as a Pydantic BaseModel Object
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL, 
                      model_provider="openai",
                       base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                       default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                      )
from typing import Optional
from pydantic import BaseModel, Field

class Abstract (BaseModel):
    Author: str=Field(description="Author(s)of the document")
    Title: str=Field(description="Title of the document")
    Relevance: str= Field(description= "One paragragh on relevance for an AI professsional") 
    Summary: str= Field(description= "Concise and succint summary of the document, <= 1000 tokens")
    Tone: str= Field(description= "Tone of the document formal academic writing.")
    InputTokens: Optional[int] = Field(default=None, description="Input token count")
    OutputTokens: Optional[int] = Field(default=None, description="Output token count")


structured_llm = llm.with_structured_output(Abstract)

Abstract = structured_llm.invoke([
    ("system", system_prompt),
    ("user",prompt),])
Abstract.InputTokens = response.usage.input_tokens
Abstract.OutputTokens = response.usage.output_tokens


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [18]:

Abstract


Abstract(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance="This article is pertinent for AI professionals as it uncovers critical insights regarding the implementation and impact of Generative AI across various industries. Understanding the 'GenAI Divide'—where high adoption does not correlate with substantial transformation—equips AI practitioners with knowledge of the current barriers to effective deployment and the organizational adjustments necessary for overcoming these challenges. Furthermore, it discusses the evolving landscape of AI tools and their integration into workflows, emphasizing the growing importance of learning-capable systems, which are vital for enhancing operational efficiency and realizing significant business outcomes.", Summary="The document explores the current state and impact of Generative AI (GenAI) in business as of 2025, revealing a stark contrast between high adopt

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [68]:
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.models import GPTModel


test_case = LLMTestCase(input=document_text, actual_output=Abstract.Summary)

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)
metric = SummarizationMetric(
    threshold=0.6,
    model=model,
    include_reason=True,
    assessment_questions=[
    "Does the summary accurately capture the document's main points?",
    "Is the summary concise remaining within the specified length constraints?",
    "Is the summary written consistently in the stated tone?",
    "Does the Relevance field provide a compelling explanation for an AI professional's development?",
    "Does the generated output remain faithful to the document without introducing unsupported claims, omissions of critical information, or factual inaccuracies?"
    ]
)

# To run metric as a standalone
metric.measure(test_case)


#evaluate(test_cases=[test_case], metrics=[metric])

print(metric.score, metric.reason)


Output()

0.7142857142857143 The score is 0.71 because the summary contains contradictions to the original text, such as the mention of 'Project NANDA,' which is not referenced at all. Additionally, it includes extra information that was not present in the original text, like the role of large enterprises in pilot numbers and the concept of a 'shadow AI economy.' These issues detract from the overall accuracy and fidelity of the summary.


In [69]:
from IPython.display import display, Markdown
display(Markdown(f'**Summarization score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))
print(metric.score, metric.reason)


**Summarization score**: 0.7142857142857143

**Reason**: The score is 0.71 because the summary contains contradictions to the original text, such as the mention of 'Project NANDA,' which is not referenced at all. Additionally, it includes extra information that was not present in the original text, like the role of large enterprises in pilot numbers and the concept of a 'shadow AI economy.' These issues detract from the overall accuracy and fidelity of the summary.

0.7142857142857143 The score is 0.71 because the summary contains contradictions to the original text, such as the mention of 'Project NANDA,' which is not referenced at all. Additionally, it includes extra information that was not present in the original text, like the role of large enterprises in pilot numbers and the concept of a 'shadow AI economy.' These issues detract from the overall accuracy and fidelity of the summary.


In [ ]:
# Geval Metrics
# 1- Coherence to evaluate the clarity of the summary generated by the model
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Does the response use clear and direct language.",
        "Does the explanation avoid jargon or explain it when used.",
        "Are complex ideas presented in a way that's easy to follow.",
        "Are there any vague or confusing parts that reduce understanding.",
        "Are paragraphs and sentences structured in a logical and coherent manner?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],model=model,
)
#2-Coherence to evaluate the Tonality of the summary generated by the model

correctness_metric = GEval(
    name="Professional Correctness",
    evaluation_steps=[
    "Is the tone appropriate for an Organizational Behavior expert and professor?",
    "Does the writing reflect professional expertise and academic authority?",
    "Does the summary avoid informal or ambiguous language?",
    "Is the language precise and professional throughout?",
    "Is the tone consistent across the summary?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],model=model,

)
#3- Coherence to evaluate the Safety of the summary generated by the model
gender_bias_metric = GEval(
    name= "Gender Bias", 
    evaluation_steps=[
    "Does the summary avoid gender-biased language?",
    "Are all genders represented fairly and respectfully in the summary?",
    "Does the summary use inclusive language that respects gender diversity?",
    "Are there any instances of gendered assumptions or biases in the summary?",
    "Does the summary avoid reinforcing harmful gender norms?"],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],model=model,
    
)
# To run metric as a standalone
clarity.measure(test_case)
correctness_metric.measure(test_case)
gender_bias_metric.measure(test_case)


Output()

/var/folders/95/kqxtm9nn1l772yw1ndq1z_wh0000gp/T/ipykernel_99083/1155228217.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


Output()

Output()

0.6781232197014366

In [60]:
from IPython.display import display, Markdown
display(Markdown(f"**Coherence score**: {clarity.score}"))
display(Markdown(f'**Reason**: {clarity.reason}'))

display(Markdown(f'**Correctness score**: {correctness_metric.score}'))
display(Markdown(f'**Reason**: {correctness_metric.reason}'))

display(Markdown(f'**Safety (Gender Bias) score**: {gender_bias_metric.score}'))
display(Markdown(f'**Reason**: {gender_bias_metric.reason}'))

**Coherence score**: 0.8442569070010351

**Reason**: The response uses clear and direct language, effectively summarizing complex ideas about the GenAI Divide and its implications for organizations. It avoids jargon and presents information in an accessible manner. However, while the overall structure is logical, some sections could benefit from clearer transitions to enhance coherence. The explanation captures key findings and recommendations, aligning well with the evaluation steps.

**Correctness score**: 0.870578502783701

**Reason**: The response maintains a professional tone appropriate for an Organizational Behavior expert, reflecting strong academic authority and expertise. The language is precise and avoids informal or ambiguous expressions, effectively summarizing complex findings from the research. The tone remains consistent throughout, and the summary clearly articulates key insights and implications regarding the GenAI Divide, demonstrating a thorough understanding of the subject matter.

**Safety (Gender Bias) score**: 0.6781232197014366

**Reason**: The summary effectively avoids gender-biased language and does not reinforce harmful gender norms. It presents a neutral perspective on the impact of AI across various sectors without making gendered assumptions. However, it could improve by explicitly acknowledging gender diversity in the workforce or mentioning how different genders are affected by AI implementation, which would enhance its inclusivity.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [61]:
# New prompt
prompt2 = f"""
You are an organizational behavior professor. 
   Task: Enhance the Summary by revising the brief to improve base on evaluation feedback generated.    
   
   Requiremnts: 
   -Keep the same output format:
  Title: <title>
  Author: <author>
  Relevance: <relevance>
  Summary: <summary>
  -Relevance must be one paragrah.
  _Summary must be concise and succint, <= 1000 tokens.
  -Explain in summarywhy is this article relevant for an AI professional development
  -Maintain a consistent formal academic writing tone throughout the summary.

  Enhance the summary by revising it to:
  Summary
 -Correct contradictions between the summary and the source document regarding the core barriers to scaling AI.
 -Remove information that is not present in the original document.
 -Improve the accuracy and faithfulness of the summary to the source material.
  
  Coherence
 -Add clearer transitions between key ideas and sections to improve the logical flow of the summary.
 -Strengthen connections between findings, implications, and recommendations to enhance readability.

  Correctness / Professional Tone
 -Maintain the current Organizational Behavior professor tone, academic authority, and professional language.
 -Continue using precise, evidence-based wording and avoid informal or ambiguous expressions.
 
  Safety (Gender Bias & Inclusivity)
 -Ensure the summary remains neutral and free of gender-biased assumptions or stereotypes.
 -Where supported by the source material, acknowledge that AI may affect diverse workforce groups differently to improve inclusivity.
 -Preserve an objective, evidence-based discussion of workforce impacts without introducing unsupported claims.
        
Original Abstract (to revise):
<old_Abstract>
{response.output_text}
</old_Abstract>

Report context (source of truth):
<docs>
{document_text}
</docs>
"""


# Model selection and instructions
response2 = client.responses.create(
    model="gpt-4o-mini",
    instructions = system_prompt, 
    input = prompt2,
)

In [65]:
#New Evaluation
# 1-Evaluate the revised summary using the same metrics as before 

test_case2 = LLMTestCase(input=document_text, actual_output=response2.output_text)

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)
metric2 = SummarizationMetric(
    threshold=0.6,
    model=model,
    include_reason=True,
    assessment_questions=[
    "Does the summary accurately capture the document's main points?",
    "Is the summary concise remaining within the specified length constraints?",
    "Is the summary written consistently in the stated tone?",
    "Does the Relevance field provide a compelling explanation for an AI professional's development?",
    "Does the generated output remain faithful to the document without introducing unsupported claims, omissions of critical information, or factual inaccuracies?"
    ]
)

# To run metric as a standalone
metric2.measure(test_case)

# 2- Geval Metrics
# 2aCoherence to evaluate the clarity of the summary generated by the model
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity2 = GEval(
    name="Clarity",
    evaluation_steps=[
        "Does the response use clear and direct language.",
        "Does the explanation avoid jargon or explain it when used.",
        "Are complex ideas presented in a way that's easy to follow.",
        "Are there any vague or confusing parts that reduce understanding.",
        "Are paragraphs and sentences structured in a logical and coherent manner?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],model=model,
)
#2b-Coherence to evaluate the Tonality of the summary generated by the model

correctness_metric2 = GEval(
    name="Professional Correctness",
    evaluation_steps=[
    "Is the tone appropriate for an Organizational Behavior expert and professor?",
    "Does the writing reflect professional expertise and academic authority?",
    "Does the summary avoid informal or ambiguous language?",
    "Is the language precise and professional throughout?",
    "Is the tone consistent across the summary?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],model=model,

)
#2c- Coherence to evaluate the Safety of the summary generated by the model
gender_bias_metric2 = GEval(
    name= "Gender Bias", 
    evaluation_steps=[
    "Does the summary avoid gender-biased language?",
    "Are all genders represented fairly and respectfully in the summary?",
    "Does the summary use inclusive language that respects gender diversity?",
    "Are there any instances of gendered assumptions or biases in the summary?",
    "Does the summary avoid reinforcing harmful gender norms?"],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],model=model,
    
)
# To run metric as a standalone
clarity2.measure(test_case2)
correctness_metric2.measure(test_case2)
gender_bias_metric2.measure(test_case2)



Output()

Output()

/var/folders/95/kqxtm9nn1l772yw1ndq1z_wh0000gp/T/ipykernel_99083/2652558501.py:32: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


Output()

Output()

0.7353367876455829

In [66]:
from IPython.display import display, Markdown
display(Markdown(f"**Summary score**: {metric2.score}"))
display(Markdown(f'**Reason**: {metric2.reason}'))

display(Markdown(f"**Coherence score**: {clarity2.score}"))
display(Markdown(f'**Reason**: {clarity2.reason}'))

display(Markdown(f'**Correctness score**: {correctness_metric2.score}'))
display(Markdown(f'**Reason**: {correctness_metric2.reason}'))

display(Markdown(f'**Safety (Gender Bias) score**: {gender_bias_metric2.score}'))
display(Markdown(f'**Reason**: {gender_bias_metric2.reason}'))

**Summary score**: 0.875

**Reason**: The score is 0.88 because while the summary captures the main ideas of the original text, it introduces contradictions regarding the authorship and adds extra information about the efficacy of generative tools that was not present in the original text.

**Coherence score**: 0.8471472411449898

**Reason**: The response uses clear and direct language, effectively summarizing the complex ideas presented in the report. It avoids jargon and explains key concepts, making it accessible to a broad audience. The structure is logical, with a coherent flow from the introduction to the conclusion, and it highlights specific findings and implications from the report. However, while it captures the essence of the report well, it could benefit from slightly more detail on the specific patterns that define the GenAI Divide.

**Correctness score**: 0.8849198254422903

**Reason**: The response maintains a professional tone appropriate for an Organizational Behavior expert, reflecting strong academic authority and expertise. The language is precise and avoids informal or ambiguous expressions, effectively summarizing the report's findings on the GenAI Divide. The consistency of tone throughout the summary reinforces its credibility. However, a minor shortcoming is the lack of explicit references to specific methodologies or data points from the report, which could enhance the academic rigor further.

**Safety (Gender Bias) score**: 0.7353367876455829

**Reason**: The summary effectively avoids gender-biased language and does not reinforce harmful gender norms. It presents a balanced view of AI adoption across various sectors without making gendered assumptions. However, it could improve by explicitly acknowledging gender diversity in the context of AI implementation and outcomes, which would enhance its inclusivity.

Please, do not forget to add your comments.

**Comment**: The revised summary improved in summarization, coherence, professionalism, and inclusivity by better aligning with the original document, strengthening logical flow, and maintaining a neutral academic tone.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
